In [1]:
from pathlib import Path
import glob

BASE_PATH = Path("/Users/sara.alsiyat/Desktop/Uni/Study/Winter/420 - Database systems/Final Project")
ON_TIME_PATH = BASE_PATH / "data" / "01_OnTime_Performance"
OUTPUT_PATH = ON_TIME_PATH / "merged"
OUTPUT_PATH.mkdir(parents=True, exist_ok=True)

YEARS = [2023, 2024, 2025]

def is_lfs_pointer(csv_path: Path) -> bool:
    """Detect Git LFS pointer files (these are NOT real CSV data)."""
    with open(csv_path, "r", encoding="utf-8", errors="replace") as f:
        first_line = f.readline().strip()
    return first_line.startswith("version https://git-lfs.github.com/spec/v1")

def merge_year_raw(year: int) -> Path | None:
    year_dir = ON_TIME_PATH / str(year)
    files = sorted(glob.glob(str(year_dir / f"OnTime_{year}_*.csv")))

    if not files:
        print(f"❌ No files found for {year}: {year_dir}")
        return None

    files = [Path(f) for f in files]

    # Fail fast if any file is an LFS pointer
    bad = [f for f in files if is_lfs_pointer(f)]
    if bad:
        print(f"\n❌ Found Git LFS pointer files for {year} (not real CSV data):")
        for f in bad[:10]:
            print(" -", f)
        print("Fix by running inside repo: git lfs install && git lfs pull && git lfs checkout")
        return None

    output_file = OUTPUT_PATH / f"OnTime_{year}_ALL.csv"
    if output_file.exists():
        output_file.unlink()  # prevent accidental duplicate/append situations

    print(f"\n📦 RAW merging {len(files)} files for {year} -> {output_file.name}")
    for f in files:
        print(" -", f.name)

    with open(output_file, "wb") as out:
        for i, f in enumerate(files):
            with open(f, "rb") as inp:
                if i == 0:
                    out.write(inp.read())      # keep header
                else:
                    inp.readline()             # skip header
                    out.write(inp.read())

    print(f"✅ Created: {output_file}")
    return output_file

# 1) Merge each year (fast)
merged_files = []
for y in YEARS:
    out = merge_year_raw(y)
    if out:
        merged_files.append(out)

# 2) Merge all years (fast)
if merged_files:
    combined_file = OUTPUT_PATH / "OnTime_2023_2025_ALL.csv"
    if combined_file.exists():
        combined_file.unlink()  # prevent duplicates if rerun

    print(f"\n📊 RAW combining all years -> {combined_file.name}")
    with open(combined_file, "wb") as out:
        for i, f in enumerate(merged_files):
            with open(f, "rb") as inp:
                if i == 0:
                    out.write(inp.read())      # keep header
                else:
                    inp.readline()             # skip header
                    out.write(inp.read())

    print(f"🎉 Created: {combined_file}")

print("\nDONE ✅")
print("Merged files are in:", OUTPUT_PATH)


📦 RAW merging 12 files for 2023 -> OnTime_2023_ALL.csv
 - OnTime_2023_1.csv
 - OnTime_2023_10.csv
 - OnTime_2023_11.csv
 - OnTime_2023_12.csv
 - OnTime_2023_2.csv
 - OnTime_2023_3.csv
 - OnTime_2023_4.csv
 - OnTime_2023_5.csv
 - OnTime_2023_6.csv
 - OnTime_2023_7.csv
 - OnTime_2023_8.csv
 - OnTime_2023_9.csv
✅ Created: /Users/sara.alsiyat/Desktop/Uni/Study/Winter/420 - Database systems/Final Project/data/01_OnTime_Performance/merged/OnTime_2023_ALL.csv

📦 RAW merging 12 files for 2024 -> OnTime_2024_ALL.csv
 - OnTime_2024_1.csv
 - OnTime_2024_10.csv
 - OnTime_2024_11.csv
 - OnTime_2024_12.csv
 - OnTime_2024_2.csv
 - OnTime_2024_3.csv
 - OnTime_2024_4.csv
 - OnTime_2024_5.csv
 - OnTime_2024_6.csv
 - OnTime_2024_7.csv
 - OnTime_2024_8.csv
 - OnTime_2024_9.csv
✅ Created: /Users/sara.alsiyat/Desktop/Uni/Study/Winter/420 - Database systems/Final Project/data/01_OnTime_Performance/merged/OnTime_2024_ALL.csv

📦 RAW merging 10 files for 2025 -> OnTime_2025_ALL.csv
 - OnTime_2025_1.csv
 - OnTi

In [2]:
from pathlib import Path

BASE_PATH = Path("/Users/sara.alsiyat/Desktop/Uni/Study/Winter/420 - Database systems/Final Project")
MERGED_PATH = BASE_PATH / "data" / "01_OnTime_Performance" / "merged"

files = [
    MERGED_PATH / "OnTime_2023_ALL.csv",
    MERGED_PATH / "OnTime_2024_ALL.csv",
    MERGED_PATH / "OnTime_2025_ALL.csv",
]

for file in files:
    print("\n" + "="*60)
    print(f"📄 Header check: {file.name}")
    print("="*60)

    with open(file, 'r', encoding='utf-8', errors='replace') as f:
        header = f.readline().strip()

    columns = header.split(",")
    print(f"Column count: {len(columns)}")
    print(columns)


📄 Header check: OnTime_2023_ALL.csv
Column count: 42
['YEAR', 'QUARTER', 'MONTH', 'DAY_OF_MONTH', 'DAY_OF_WEEK', 'FL_DATE', 'OP_UNIQUE_CARRIER', 'OP_CARRIER_AIRLINE_ID', 'TAIL_NUM', 'OP_CARRIER_FL_NUM', 'ORIGIN_AIRPORT_ID', 'ORIGIN', 'ORIGIN_CITY_NAME', 'ORIGIN_STATE_ABR', 'ORIGIN_STATE_NM', 'DEST_AIRPORT_ID', 'DEST', 'DEST_CITY_NAME', 'DEST_STATE_ABR', 'DEST_STATE_NM', 'CRS_DEP_TIME', 'DEP_TIME', 'DEP_DELAY', 'DEP_DELAY_NEW', 'DEP_DEL15', 'CRS_ARR_TIME', 'ARR_TIME', 'ARR_DELAY', 'ARR_DELAY_NEW', 'ARR_DEL15', 'CANCELLED', 'CANCELLATION_CODE', 'DIVERTED', 'CRS_ELAPSED_TIME', 'ACTUAL_ELAPSED_TIME', 'AIR_TIME', 'DISTANCE', 'CARRIER_DELAY', 'WEATHER_DELAY', 'NAS_DELAY', 'SECURITY_DELAY', 'LATE_AIRCRAFT_DELAY']

📄 Header check: OnTime_2024_ALL.csv
Column count: 42
['YEAR', 'QUARTER', 'MONTH', 'DAY_OF_MONTH', 'DAY_OF_WEEK', 'FL_DATE', 'OP_UNIQUE_CARRIER', 'OP_CARRIER_AIRLINE_ID', 'TAIL_NUM', 'OP_CARRIER_FL_NUM', 'ORIGIN_AIRPORT_ID', 'ORIGIN', 'ORIGIN_CITY_NAME', 'ORIGIN_STATE_ABR', 'ORIGIN_

In [3]:
import pandas as pd

path = "/Users/sara.alsiyat/Desktop/Uni/Study/Winter/420 - Database systems/Final Project/data/01_OnTime_Performance/merged/OnTime_2023_2025_ALL.csv"

years = set()
months = set()
quarters = set()
year_month_pairs = set()

print("🔎 Scanning file in chunks...")

for chunk in pd.read_csv(
    path,
    engine="python",
    on_bad_lines="skip",
    chunksize=500_000
):
    years.update(chunk['YEAR'].dropna().unique())
    months.update(chunk['MONTH'].dropna().unique())
    quarters.update(chunk['QUARTER'].dropna().unique())

    year_month_pairs.update(
        zip(chunk['YEAR'], chunk['MONTH'])
    )

print("\n============================")
print("📅 DISTINCT YEARS:")
print(sorted(years))

print("\n📆 DISTINCT QUARTERS:")
print(sorted(quarters))

print("\n🗓 DISTINCT MONTHS:")
print(sorted(months))

print("\n📊 DISTINCT YEAR-MONTH COMBINATIONS:")
print(sorted(year_month_pairs))

print("\nDONE ✅")

🔎 Scanning file in chunks...

📅 DISTINCT YEARS:
[np.int64(2023), np.int64(2024), np.int64(2025)]

📆 DISTINCT QUARTERS:
[np.int64(1), np.int64(2), np.int64(3), np.int64(4)]

🗓 DISTINCT MONTHS:
[np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12)]

📊 DISTINCT YEAR-MONTH COMBINATIONS:
[(2023, 1), (2023, 2), (2023, 3), (2023, 4), (2023, 5), (2023, 6), (2023, 7), (2023, 8), (2023, 9), (2023, 10), (2023, 11), (2023, 12), (2024, 1), (2025, 1), (2025, 2), (2025, 3), (2025, 4), (2025, 5), (2025, 6), (2025, 7), (2025, 8), (2025, 9), (2025, 10)]

DONE ✅
